# Week 2 · MAP and entropy, in depth

Your log said four things here were unclear: what $\arg\max_\theta P(\theta \mid \text{data})$ actually means, how a Beta distribution works as a prior, how MAP connects to ridge regression, and how entropy, cross-entropy and KL divergence fit together. This notebook takes them one at a time, slowly, with every step written out. **Supplementary reading:** *Mathematics for Machine Learning* §8.3 (MLE and MAP) and §9.2 (MAP for linear regression = regularization), and StatQuest's videos on maximum likelihood and on ridge regression.

**How to use this notebook**
- Run cells one at a time with **Shift + Enter**.
- Before running an example, **guess what it prints**. The real result is written after `# →` in the comments, so you can check yourself.
- **Your turn** cells are for you. Open **Show one answer** only after a real attempt.
- If things get weird, click **Restart** at the top and run from the top again.

## Words you'll see
- **$\max_\theta f(\theta)$**: The **biggest value** that $f$ reaches. A number like $5$.
- **$\arg\max_\theta f(\theta)$**: The **input** $\theta$ where that biggest value happens. Read it as "the $\theta$ that maximizes $f$".
- **prior $P(\theta)$**: What you believe about $\theta$ **before** seeing data.
- **likelihood $P(\text{data} \mid \theta)$**: How probable the observed data is **if** the parameter were $\theta$.
- **posterior $P(\theta \mid \text{data})$**: What you believe about $\theta$ **after** seeing the data.
- **$\propto$**: "Proportional to": equal up to a constant factor that doesn't depend on $\theta$.

## 1. What $\arg\max$ means

**$\arg\max$ returns where the maximum is, not how big it is.**

**Example.** Suppose $f(0) = 1$, $f(1) = 5$, $f(2) = 3$:

| $\theta$ | $f(\theta)$ |
|---|---|
| 0 | 1 |
| **1** | **5** ← biggest |
| 2 | 3 |

$$
\max_\theta f(\theta) = 5 \qquad\qquad \arg\max_\theta f(\theta) = 1
$$

So **$\hat\theta = \arg\max_\theta P(\theta \mid \text{data})$** is an instruction: *try every possible value of $\theta$, compute how believable it is after seeing the data, and keep the $\theta$ with the highest score.* You did exactly this in Week 1 with `p[np.argmax(log_L)]`: `argmax` gives the **position**, and `p[...]` turns the position into the $\theta$ value. Your own comment said it: "the position where the value is the max".

In [ ]:
import numpy as np
theta = np.array([0, 1, 2]); f = np.array([1, 5, 3])
print(f.max(), np.argmax(f), theta[np.argmax(f)])    # → 5 1 1   max value, its position, the θ there

**Two rules that make $\arg\max$ easy to work with:**

| Rule | Why it's true | Why it helps |
|---|---|---|
| $\arg\max_\theta f(\theta) = \arg\max_\theta \log f(\theta)$ | $\log$ is increasing, so it never changes which input is biggest | products become sums |
| $\arg\max_\theta c \cdot f(\theta) = \arg\max_\theta f(\theta)$ for any $c > 0$ | scaling every value by the same amount keeps the order | constants like $P(\text{data})$ can be dropped |

## 2. The Beta distribution: a belief about a probability

**A Beta distribution describes how plausible each value of an unknown probability $p \in [0, 1]$ is.**

**The setting.** The unknown is a probability itself, like the chance a coin lands heads. A prior for it must live on $[0, 1]$. The Beta has two shape parameters $a, b > 0$:

$$
\text{Beta}(p \mid a, b) \;\propto\; p^{\,a-1}\, (1 - p)^{\,b-1}, \qquad 0 \le p \le 1
$$

Read it as: $p^{a-1}$ pushes the belief toward 1, and $(1-p)^{b-1}$ pushes it toward 0. **The easiest way to think about it:** $\text{Beta}(a, b)$ behaves like having **already seen $a - 1$ heads and $b - 1$ tails** before the experiment starts. These are called **pseudo-counts**.

| Prior | Like having seen… | Mean $\frac{a}{a+b}$ | Mode $\frac{a-1}{a+b-2}$ | Spread (sd) | In words |
|---|---|---|---|---|---|
| $\text{Beta}(1, 1)$ | 0 heads, 0 tails | 0.5 | (none: flat) | 0.289 | no opinion: every $p$ equally plausible |
| $\text{Beta}(2, 2)$ | 1 head, 1 tail | 0.5 | 0.5 | 0.224 | mild belief the coin is fair |
| $\text{Beta}(5, 5)$ | 4 heads, 4 tails | 0.5 | 0.5 | 0.151 | fairly sure it's fair |
| $\text{Beta}(20, 20)$ | 19 heads, 19 tails | 0.5 | 0.5 | 0.078 | very sure it's fair |
| $\text{Beta}(2, 5)$ | 1 head, 4 tails | 0.286 | 0.2 | 0.160 | believe heads is rare |

Bigger $a + b$ means a narrower curve, i.e. a **stronger** prior. Plot a few to see the shapes:

In [ ]:
import matplotlib.pyplot as plt
from math import gamma
def beta_pdf(p, a, b):
    return p**(a - 1) * (1 - p)**(b - 1) * gamma(a + b) / (gamma(a) * gamma(b))
p = np.linspace(0.001, 0.999, 400)
for a, b in [(1, 1), (2, 2), (5, 5), (20, 20), (2, 5)]:
    plt.plot(p, beta_pdf(p, a, b), label=f"Beta({a},{b})")
plt.xlabel("p (probability of heads)"); plt.ylabel("density"); plt.legend(); plt.show()

## 3. MAP for a coin, every step

**MAP picks the parameter value that is most believable after combining the prior with the data.**

**The setting.** $n$ flips with $h$ heads. Prior $\text{Beta}(a, b)$. We want $\hat p_{\text{MAP}} = \arg\max_p P(p \mid \text{data})$.

**Step 1, Bayes' rule.** $P(\text{data})$ doesn't depend on $p$, so by rule 2 of section 1 we can drop it:

$$
P(p \mid \text{data}) = \frac{P(\text{data} \mid p)\, P(p)}{P(\text{data})} \;\propto\; \underbrace{p^{h} (1-p)^{n-h}}_{\text{likelihood}} \;\cdot\; \underbrace{p^{a-1} (1-p)^{b-1}}_{\text{Beta prior}}
$$

**Step 2, multiply.** Same bases, so add the exponents:

$$
P(p \mid \text{data}) \;\propto\; p^{\,h + a - 1}\, (1-p)^{\,n - h + b - 1} \quad=\quad \text{the shape of } \text{Beta}(h + a,\; n - h + b)
$$

So **the posterior is another Beta**. The data's counts simply add to the pseudo-counts. That's why Beta is the standard prior for a probability: it's **conjugate** to the coin-flip likelihood.

**Step 3, take the log** (rule 1 of section 1):

$$
\log P(p \mid \text{data}) = (h + a - 1) \log p + (n - h + b - 1) \log(1 - p) + \text{const}
$$

**Step 4, set the derivative to 0 and solve:**

$$
\frac{h + a - 1}{p} - \frac{n - h + b - 1}{1 - p} = 0 \;\Longrightarrow\; (h + a - 1)(1 - p) = (n - h + b - 1)\, p
$$

$$
h + a - 1 = p\,\big[(h + a - 1) + (n - h + b - 1)\big] = p\,(n + a + b - 2) \;\Longrightarrow\; \boxed{\hat p_{\text{MAP}} = \frac{h + a - 1}{n + a + b - 2}}
$$

In plain words: **it's the MLE $\frac{h}{n}$ computed as if you had $a - 1$ extra heads and $b - 1$ extra tails.**

**Worked example: 7 heads in 10, prior $\text{Beta}(2, 2)$.** The posterior is $\text{Beta}(9, 5)$, and $\hat p_{\text{MAP}} = \frac{7 + 1}{10 + 2} = \frac{8}{12} \approx 0.667$. To see $\arg\max$ literally, evaluate prior × likelihood at a few values of $p$ and pick the biggest:

| $p$ | prior $\text{Beta}(2,2)$ | likelihood $p^7(1-p)^3$ | prior × likelihood | verdict |
|---|---|---|---|---|
| 0.500 | 1.500 | 0.000977 | 0.001465 |  |
| 0.600 | 1.440 | 0.001792 | 0.002580 |  |
| 0.650 | 1.365 | 0.002102 | 0.002869 |  |
| **0.667** | 1.333 | 0.002168 | **0.002890** | ← biggest: this is the MAP |
| 0.700 | 1.260 | **0.002224** | 0.002802 | ← biggest likelihood: this is the MLE |
| 0.800 | 0.960 | 0.001678 | 0.001611 |  |

Look at the row for $0.7$: it has the highest **likelihood**, but the prior gives it slightly less weight than $0.667$, so the product peaks a little closer to $0.5$. **The prior pulls the estimate toward what you believed beforehand.**

In [ ]:
p = np.linspace(0.001, 0.999, 99901)
posterior = p**8 * (1 - p)**4           # Beta(9, 5) shape: prior × likelihood
likelihood = p**7 * (1 - p)**3
print(round(p[np.argmax(posterior)], 3), round(p[np.argmax(likelihood)], 3))   # → 0.667 0.7

**How much the prior matters depends on how much data you have:**

| Data | MLE $h/n$ | MAP, $\text{Beta}(2,2)$ | MAP, $\text{Beta}(20,20)$ |
|---|---|---|---|
| 7 / 10 | 0.7 | 0.6667 | 0.5417 |
| 70 / 100 | 0.7 | 0.6961 | 0.6449 |
| 700 / 1000 | 0.7 | 0.6996 | 0.6927 |

A weak prior barely moves the estimate. A strong prior moves it a lot when data is scarce. **As $n$ grows, MAP → MLE**, because 1000 real flips outweigh 38 imaginary ones.

**Drill (on paper).** 12 flips, 3 heads, prior $\text{Beta}(3, 3)$. Write the posterior distribution, then the MAP estimate.

<details><summary>Worked answer</summary>

Posterior: $\text{Beta}(3 + 3,\ 9 + 3) = \text{Beta}(6, 12)$. MAP is its mode: $\frac{6 - 1}{6 + 12 - 2} = \frac{5}{16} = 0.3125$. That's the same as $\frac{h + a - 1}{n + a + b - 2} = \frac{3 + 2}{12 + 4}$.

</details>

## 4. MAP with a Gaussian prior is ridge regression

**Ridge regression is MAP estimation for linear regression when the prior on each weight is a Gaussian centered at 0.**

**The setting.** One feature, no intercept to keep the algebra short: $y_i = w x_i + \varepsilon_i$ with noise $\varepsilon_i \sim \mathcal{N}(0, \sigma^2)$. The prior says weights are probably small: $w \sim \mathcal{N}(0, \tau^2)$.

**Step 1, the two pieces.** Each Gaussian contributes $\exp(-\frac{(\text{value} - \text{mean})^2}{2 \cdot \text{variance}})$, so taking logs gives squares:

$$
\log \underbrace{P(\text{data} \mid w)}_{\text{likelihood}} = -\frac{1}{2\sigma^2} \sum_{i=1}^{n} (y_i - w x_i)^2 + \text{const} \qquad \log \underbrace{P(w)}_{\text{prior}} = -\frac{w^2}{2\tau^2} + \text{const}
$$

**Step 2, MAP = maximize log likelihood + log prior.** Maximizing something is the same as minimizing its negative:

$$
\hat w_{\text{MAP}} = \arg\min_w \left[ \frac{1}{2\sigma^2} \sum_i (y_i - w x_i)^2 + \frac{1}{2\tau^2} w^2 \right]
$$

**Step 3, multiply by $2\sigma^2$.** This doesn't move the minimum (rule 2 of section 1):

$$
\hat w_{\text{MAP}} = \arg\min_w \Big[ \underbrace{\sum_i (y_i - w x_i)^2}_{\text{least squares}} + \underbrace{\lambda\, w^2}_{\text{ridge penalty}} \Big] \qquad \text{with } \lambda = \frac{\sigma^2}{\tau^2}
$$

**That's exactly ridge regression.** The penalty strength $\lambda$ is the noise variance divided by the prior variance. A **tight prior** (small $\tau$, "I'm sure the weights are near 0") means a **big $\lambda$** and strong shrinkage. A **vague prior** ($\tau \to \infty$) means $\lambda \to 0$, which is plain least squares, i.e. the MLE. It's the same pattern as the coin.

**Step 4, solve.** Take the derivative and set it to 0:

$$
-2 \sum_i x_i (y_i - w x_i) + 2\lambda w = 0 \;\Longrightarrow\; w \Big( \sum_i x_i^2 + \lambda \Big) = \sum_i x_i y_i \;\Longrightarrow\; \hat w = \frac{\sum_i x_i y_i}{\sum_i x_i^2 + \lambda}
$$

**Worked example.** $x = (1, 2, 3)$, $y = (1, 2, 2)$. Then $\sum x_i y_i = 1 + 4 + 6 = 11$ and $\sum x_i^2 = 1 + 4 + 9 = 14$:

| $\lambda$ | $\hat w = \frac{11}{14 + \lambda}$ | Meaning |
|---|---|---|
| 0 | 0.786 | no prior: ordinary least squares (the MLE) |
| 1 | 0.733 | weak prior: a little shrinkage |
| 14 | 0.393 | strong prior: the weight is halved |

In [ ]:
x = np.array([1.0, 2.0, 3.0]); y = np.array([1.0, 2.0, 2.0])
for lam in [0, 1, 14]:
    print(lam, round((x @ y) / (x @ x + lam), 3))
# → 0 0.786
#   1 0.733
#   14 0.393

**With many features** the same steps give $\hat w = (X^\top X + \lambda I)^{-1} X^\top y$. The $+\lambda I$ also makes the matrix invertible even when features are copies of each other (notebook `01`, §3).

|  | Coin | Linear regression |
|---|---|---|
| MLE | $\frac{h}{n}$ | least squares: $(X^\top X)^{-1} X^\top y$ |
| prior | $\text{Beta}(a, b)$ | $w_j \sim \mathcal{N}(0, \tau^2)$ |
| MAP | $\frac{h + a - 1}{n + a + b - 2}$ | ridge: $(X^\top X + \lambda I)^{-1} X^\top y$, $\lambda = \sigma^2 / \tau^2$ |
| effect of the prior | pulls toward the prior's center, 0.5 | pulls the weights toward 0 |
| more data | MAP → MLE | the penalty matters less and less |

A **Laplace** prior instead of a Gaussian gives the penalty $\lambda \sum_j \lvert w_j \rvert$, which is the **lasso** (notebook `04`).

## 5. Entropy

**Entropy is the average surprise of a random outcome: how unpredictable it is.**

**The setting.** A discrete random variable with outcomes $x_1, \dots, x_k$ and probabilities $p_1, \dots, p_k$. Only the probabilities matter, not what the outcomes are. Real examples: class counts in a dataset, letter frequencies, which of 3 machines produced a part.

$$
H(p) = -\sum_{i=1}^{k} p_i \log p_i = \mathbb{E}\big[-\log p(X)\big]
$$

Here $p_i$ is the probability of outcome $i$, and $-\log p_i$ is the **surprise** of seeing it: a sure thing ($p_i = 1$) has surprise 0, and a rare thing has a big surprise. Entropy is the probability-weighted average surprise. With $\log_2$ the unit is **bits**, and with $\ln$ it's **nats**.

**Worked example: a fair coin**, written out term by term:

$$
H = -\big[\underbrace{0.5 \log_2 0.5}_{\text{heads}} + \underbrace{0.5 \log_2 0.5}_{\text{tails}}\big] = -\big[0.5 \cdot (-1) + 0.5 \cdot (-1)\big] = 1 \text{ bit}
$$

| Distribution | $p$ | Entropy (bits) | Note |
|---|---|---|---|
| fair coin | (0.5, 0.5) | 1.000 | the most uncertain 2-outcome case |
| biased coin | (0.9, 0.1) | 0.469 | mostly predictable |
| always heads | (1, 0) | 0 | no surprise at all (uses $0 \log 0 = 0$) |
| fair die | $(\frac16, \dots, \frac16)$ | 2.585 | $= \log_2 6$ |

**Key properties.**

| Property | Statement |
|---|---|
| range | $0 \le H(p) \le \log k$ |
| minimum | $H = 0$ only when one outcome is certain |
| maximum | $H = \log k$ only for the uniform distribution |
| convention | $0 \log 0 = 0$ (an impossible outcome adds nothing) |

**Where it's used:** decision trees choose splits that reduce entropy most (information gain, Week 3); it's the first half of cross-entropy, below; and it measures how mixed the classes in a dataset are.

In [ ]:
H = lambda p: sum(-pi * np.log2(pi) for pi in p if pi > 0)     # skip 0 terms: 0 log 0 = 0
print(H([0.5, 0.5]), round(H([0.9, 0.1]), 3), H([1, 0]), round(H([1/6] * 6), 3))   # → 1.0 0.469 0.0 2.585

**Connection:** entropy uses the true $p$ for both the weights *and* the surprise. Cross-entropy keeps the true weights but measures surprise with the model's probabilities $q$.

## 6. Cross-entropy loss

**Cross-entropy measures how much probability the model gave to what actually happened: less probability on the right answer means more loss.**

**The setting.** $p$ is the true distribution over $k$ classes; for one labeled example it's **one-hot**, e.g. $p = (1, 0, 0)$ for "cat". $q$ is the model's predicted probabilities, which come from a softmax.

$$
H(p, q) = -\sum_{i=1}^{k} p_i \log q_i
$$

**The simplification, term by term.** With $p = (1, 0, 0)$ and $q = (0.6, 0.3, 0.1)$:

$$
H(p, q) = -\big[\underbrace{1 \cdot \ln 0.6}_{\text{cat}} + \underbrace{0 \cdot \ln 0.3}_{\text{dog}} + \underbrace{0 \cdot \ln 0.1}_{\text{bird}}\big] = -\ln 0.6 \approx 0.511
$$

The zeros in $p$ wipe out every term except the true class $c$. So for one example, **cross-entropy $= -\ln q_c$**, the negative log of the probability given to the correct class.

| $q_{\text{cat}}$ (true class: cat) | loss $-\ln q_{\text{cat}}$ | In words |
|---|---|---|
| 0.90 | 0.105 | confident and right: tiny loss |
| 0.70 | 0.357 |  |
| 0.50 | 0.693 | a coin flip between options |
| 0.20 | 1.609 |  |
| 0.01 | 4.605 | confident and wrong: huge loss |

**Binary form** (logistic regression, notebook `05`). With label $y \in \{0, 1\}$ and predicted $\hat p = P(y = 1)$:

$$
L = -\big[y \log \hat p + (1 - y) \log(1 - \hat p)\big]
$$

If $y = 1$ only the first term survives, giving $-\log \hat p$. If $y = 0$ only the second survives, giving $-\log(1 - \hat p)$. Over a dataset, the loss is the **average** over the $n$ examples.

**Where it comes from:** minimizing cross-entropy is the same as maximizing the likelihood of the labels, exactly like the coin MLE in Week 1. The labels are coin flips (binary) or die rolls (multi-class), and $q$ is the model's guess at their probabilities.

**What it is, and what it isn't:**

| Thing | What it is | Example |
|---|---|---|
| cross-entropy loss | **one number** per example (or averaged over the batch) | $0.511$ |
| gradient w.r.t. the logits $z$ | a vector: how to nudge each score | $q - p = (0.6 - 1,\ 0.3,\ 0.1) = (-0.4, 0.3, 0.1)$ |
| gradient w.r.t. the weights $\theta$ | a vector or matrix, the same size as the weights | what gradient descent uses |

**Practical note:** libraries combine softmax and cross-entropy into one function that takes the raw scores (logits), e.g. PyTorch's `nn.CrossEntropyLoss`, because computing them together is numerically stable. It's the same trick as subtracting the max in your Week 1 softmax.

**Connection:** cross-entropy is always at least the entropy, $H(p, q) \ge H(p)$. The gap is exactly the KL divergence.

## 7. KL divergence

**KL divergence measures how different a distribution $q$ is from a reference distribution $p$: the extra surprise you pay for using $q$ when the data really comes from $p$.**

$$
D_{\mathrm{KL}}(p \,\Vert\, q) = \sum_{i} p_i \log \frac{p_i}{q_i}
$$

Read $D_{\mathrm{KL}}(p \Vert q)$ as "the divergence of $q$ from $p$"; $p$ is the truth and $q$ is the approximation.

**Where it comes from,** in one line. Use $\log \frac{p_i}{q_i} = \log p_i - \log q_i$ and split the sum:

$$
D_{\mathrm{KL}}(p \Vert q) = \sum_i p_i \log p_i - \sum_i p_i \log q_i = \underbrace{-H(p)}_{\text{minus entropy}} + \underbrace{H(p, q)}_{\text{cross-entropy}}
$$

$$
\boxed{H(p, q) = H(p) + D_{\mathrm{KL}}(p \Vert q)}
$$

In plain words: **cross-entropy = the unavoidable surprise (entropy) + the extra surprise from the wrong model (KL).** When training a classifier, $H(p)$ is fixed by the data, so minimizing cross-entropy is the same as minimizing KL.

**Worked example: weather.** True $p = (0.5, 0.3, 0.2)$ for (sun, cloud, rain), forecast $q = (0.4, 0.4, 0.2)$, natural log:

| Outcome | $p_i$ | $q_i$ | $p_i \ln \frac{p_i}{q_i}$ | Note |
|---|---|---|---|---|
| sun | 0.5 | 0.4 | 0.1116 | $q$ under-predicts sun: positive |
| cloud | 0.3 | 0.4 | −0.0863 | $q$ over-predicts cloud: a single term can be negative |
| rain | 0.2 | 0.2 | 0 | a perfect match contributes nothing |
| **total** |  |  | **0.0253** | the total is never negative |

Check with the identity: $H(p) = 1.0297$ nats and $H(p, q) = 1.0549$ nats, and $1.0549 - 1.0297 = 0.0253$. It matches.

In [ ]:
p = np.array([0.5, 0.3, 0.2]); q = np.array([0.4, 0.4, 0.2])
terms = p * np.log(p / q)
print(terms.round(4), round(terms.sum(), 4))                          # → [ 0.1116 -0.0863  0.    ] 0.0253
print(round(-(p * np.log(q)).sum() + (p * np.log(p)).sum(), 4))      # → 0.0253   H(p,q) - H(p)
print(round((q * np.log(q / p)).sum(), 4))                            # → 0.0258   the other direction differs

**Key properties.**

| Property | Statement |
|---|---|
| never negative | $D_{\mathrm{KL}}(p \Vert q) \ge 0$ |
| zero exactly when | $q = p$ |
| not symmetric | $D_{\mathrm{KL}}(p \Vert q) \ne D_{\mathrm{KL}}(q \Vert p)$ in general ($0.0253$ vs $0.0258$ above), so it isn't a distance |
| can be infinite | if $q_i = 0$ for an outcome with $p_i > 0$: the model said "impossible" and it happened |

**Where it's used:** training classifiers (via cross-entropy), VAEs (Week 7 keeps the latent distribution close to $\mathcal{N}(0, 1)$ with a KL term), Bayesian inference, and the KL penalty that keeps a fine-tuned LLM close to its original model.

**Continuous version (for Week 7):** for two normals with the same $\sigma$, $D_{\mathrm{KL}}\big(\mathcal{N}(\mu_1, \sigma^2) \Vert \mathcal{N}(\mu_2, \sigma^2)\big) = \frac{(\mu_1 - \mu_2)^2}{2\sigma^2}$.

**Drill (on paper).** $p = (0.5, 0.5)$ and $q = (0.75, 0.25)$. Compute $H(p)$, $H(p, q)$ and $D_{\mathrm{KL}}(p \Vert q)$ in bits, and check the identity.

<details><summary>Worked answer</summary>

$H(p) = 1$ bit. $H(p, q) = -[0.5 \log_2 0.75 + 0.5 \log_2 0.25] = -[0.5 \cdot (-0.415) + 0.5 \cdot (-2)] = 1.208$ bits. $D_{\mathrm{KL}} = 0.5 \log_2 \frac{0.5}{0.75} + 0.5 \log_2 \frac{0.5}{0.25} = 0.5 \cdot (-0.585) + 0.5 \cdot 1 = 0.208$ bits. Identity: $1 + 0.208 = 1.208$. ✓

</details>

---
Next: `03_shapes_and_slicing.ipynb`. Then the ML lessons start with `04`.